In [1]:
import pickle
import pandas as pd

from pathlib import Path

from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense
)

from tensorflow.keras.models import Model

print("🚀 HelpDesk AI - Building Model")

🚀 HelpDesk AI - Building Model


In [2]:
PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = PROJECT_ROOT / "saved_models"

with open(MODEL_DIR / "tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 11496


In [3]:
MAX_LENGTH = 64

EMBEDDING_DIM = 64
LSTM_UNITS = 128

In [4]:
encoder_inputs = Input(
    shape=(MAX_LENGTH,),
    name="encoder_inputs"
)

encoder_embedding = Embedding(
    input_dim=vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="encoder_embedding"
)(encoder_inputs)

encoder_lstm = LSTM(
    LSTM_UNITS,
    return_state=True,
    name="encoder_lstm"
)

_, state_h, state_c = encoder_lstm(
    encoder_embedding
)

encoder_states = [state_h, state_c]

In [5]:
decoder_inputs = Input(
    shape=(MAX_LENGTH,),
    name="decoder_inputs"
)

decoder_embedding = Embedding(
    input_dim=vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="decoder_embedding"
)(decoder_inputs)

decoder_lstm = LSTM(
    LSTM_UNITS,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

In [6]:
decoder_dense = Dense(
    vocab_size,
    activation="softmax",
    name="decoder_output"
)

decoder_outputs = decoder_dense(
    decoder_outputs
)

In [7]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

In [8]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 64, 64)    │    735,744 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 64)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 64, 64)    │    735,744 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │     98,816 │ encoder_embeddin… │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 64, 128), │     98,816 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, 64, 11496) │  1,482,984 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,152,104 (12.02 MB)

 Trainable params: 3,152,104 (12.02 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "saved_models"

MODEL_DIR.mkdir(exist_ok=True)

model.save(MODEL_DIR / "chatbot_model.keras")

print("✅ Model saved!")

✅ Model saved!
